# 00 — Exploration du dataset

Objectifs (semaine 1) :
- Vérifier la structure `data/clean` / `data/noisy`
- Compter le nombre de paires alignées
- Distribution des durées des clips
- Écouter / regarder quelques exemples

## ⚠️ Setup — toujours exécuter cette cellule en premier

Chaque notebook Colab a son **propre kernel** : le `sys.path` de `main.ipynb` n'est pas partagé.
Cette cellule ajoute le repo au `sys.path` pour que `from src import ...` fonctionne.

In [ ]:
import os, sys
REPO_DIR = '/content/Filtre-Voix-DL'
assert os.path.exists(REPO_DIR), (
    f'{REPO_DIR} introuvable — exécute la cellule clone de main.ipynb '
    'pour cloner/mettre à jour le repo sur ce runtime Colab.'
)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

try:
    from IPython import get_ipython
    ipy = get_ipython()
    if ipy is not None:
        ipy.run_line_magic('load_ext', 'autoreload')
        ipy.run_line_magic('autoreload', '2')
except Exception as e:
    pass  # bug connu Colab Python 3.12, sans impact

print(f'sys.path OK — repo : {REPO_DIR}')

In [ ]:
import os
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
from IPython.display import Audio, display

from src import config
from src.dataset import list_pairs
from src import audio as A

## 1. Inventaire des fichiers

In [ ]:
noisy_files = sorted(Path(config.DATA_NOISY).glob('*'))
clean_files = sorted(Path(config.DATA_CLEAN).glob('*'))
print(f'Fichiers dans noisy/ : {len(noisy_files)}')
print(f'Fichiers dans clean/ : {len(clean_files)}')
if noisy_files:
    print(f'Exemples noisy : {[f.name for f in noisy_files[:3]]}')
if clean_files:
    print(f'Exemples clean : {[f.name for f in clean_files[:3]]}')

In [ ]:
# Stratégie d'appariement : essaie par nom d'abord, bascule sur index si besoin
pairs_by_name  = list_pairs(config.DATA_NOISY, config.DATA_CLEAN, pair_by='name')
pairs_by_index = list_pairs(config.DATA_NOISY, config.DATA_CLEAN, pair_by='index')

print(f'Paires par NOM   : {len(pairs_by_name)}')
print(f'Paires par INDEX : {len(pairs_by_index)}')

if len(pairs_by_name) == 0 and len(pairs_by_index) > 0:
    print('\n→ Les fichiers noisy/clean ont des noms différents.')
    print('  On utilise pair_by="index" : appariement par ordre alphabétique trié.')
    print('  Vérifie que noisy[0] correspond bien à clean[0] avant de continuer !')

pairs = pairs_by_name if pairs_by_name else pairs_by_index
print(f'\nPaires utilisées : {len(pairs)}')

## 2. Distribution des durées et sample rates

In [ ]:
if not pairs:
    print('Aucune paire — vérifie que data/clean et data/noisy contiennent des fichiers audio.')
else:
    durations = []
    sample_rates = Counter()
    channels = Counter()

    for _, noisy_path, _ in pairs:
        info = sf.info(noisy_path)
        durations.append(info.duration)
        sample_rates[info.samplerate] += 1
        channels[info.channels] += 1

    durations = np.array(durations)
    print(f'Durée  : min={durations.min():.2f}s | médiane={np.median(durations):.2f}s | '
          f'moy={durations.mean():.2f}s | max={durations.max():.2f}s')
    print(f'Sample rates : {dict(sample_rates)}')
    print(f'Canaux       : {dict(channels)}')

In [ ]:
if len(pairs) > 0:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.hist(durations, bins=30, color='steelblue', edgecolor='black')
    ax.axvline(config.CLIP_DURATION, color='red', linestyle='--',
               label=f'CLIP_DURATION = {config.CLIP_DURATION}s')
    ax.set_xlabel('Durée (s)')
    ax.set_ylabel('Nombre de paires')
    ax.set_title('Distribution des durées')
    ax.legend()
    plt.tight_layout()
    plt.show()

    pct_above = 100 * (durations >= config.CLIP_DURATION).mean()
    print(f'{pct_above:.1f}% des clips font au moins {config.CLIP_DURATION}s (pas de padding nécessaire).')

## 3. Aperçu d'une paire au hasard

In [ ]:
if not pairs:
    print('Pas de paires disponibles.')
else:
    idx = np.random.randint(0, len(pairs))
    label, noisy_path, clean_path = pairs[idx]
    print(f'Paire #{idx} : {label}')
    print(f'  noisy : {noisy_path}')
    print(f'  clean : {clean_path}')

    noisy = A.load_audio(noisy_path)
    clean = A.load_audio(clean_path)

    print('Noisy :')
    display(Audio(noisy, rate=config.SAMPLE_RATE))
    print('Clean :')
    display(Audio(clean, rate=config.SAMPLE_RATE))

    A.plot_pair(noisy, clean)
    plt.show()